In [13]:
import tensorflow as tf
from pathlib import Path
import statistics

In [23]:
train_accuracy_arr = []
train_loss_arr = []
val_accuracy_arr = []
val_loss_arr = []

for i in range(10):
    tf.random.set_seed(123)

    project_dir = Path.cwd().parent
    train_dir = project_dir / "img/train"
    validation_dir = project_dir / "img/validate"

    img_height = 224
    img_width = 224
    batch_size = 32

    train_dataset = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        image_size=(img_height, img_width),
        batch_size=batch_size
    )

    validation_dataset = tf.keras.utils.image_dataset_from_directory(
        validation_dir,
        image_size=(img_height, img_width),
        batch_size=batch_size
    )

    class_names = train_dataset.class_names

    AUTOTUNE = tf.data.AUTOTUNE

    train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

    """ 
    I created another notebook to give EfficientNetB0 architecture a try but there seems 
    to be an issue with my versions of tf and keras as I tried many different tweaks and 
    kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
    like the best fit for my small dataset if I want to use transfer learning
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(img_height, img_width, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    @tf.keras.utils.register_keras_serializable()
    class MobileNetV2Preprocess(tf.keras.layers.Layer):
        def call(self, x):
            return tf.keras.applications.mobilenet_v2.preprocess_input(x)

    model = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        MobileNetV2Preprocess(),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=['accuracy']
    )

    model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=15
    )

    train_loss, train_acc = model.evaluate(train_dataset, verbose=0)
    val_loss, val_acc = model.evaluate(validation_dataset, verbose=0)

    model.save('../models/mobilenetv2_model.keras')

    train_accuracy_arr.append(train_acc)
    train_loss_arr.append(train_loss)
    val_accuracy_arr.append(val_acc)
    val_loss_arr.append(val_loss)



Found 161 files belonging to 3 classes.


Found 42 files belonging to 3 classes.
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 882ms/step - accuracy: 0.3851 - loss: 1.5506 - val_accuracy: 0.5000 - val_loss: 1.0206
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 168ms/step - accuracy: 0.5404 - loss: 1.0356 - val_accuracy: 0.6429 - val_loss: 0.7072
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step - accuracy: 0.6149 - loss: 0.9387 - val_accuracy: 0.8095 - val_loss: 0.4946
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - accuracy: 0.6522 - loss: 0.9488 - val_accuracy: 0.8571 - val_loss: 0.4272
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.6957 - loss: 0.8735 - val_accuracy: 0.8571 - val_loss: 0.3494
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.7329 - loss: 0.7041 - val_accuracy: 0.8095 - val_loss: 0.4481
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.7019 - loss: 0.7553 - val_accuracy: 0.8571 - val_loss: 0.3406
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.7578 - loss: 0

In [25]:
avg_train_accuracy = statistics.mean(train_accuracy_arr) * 100
avg_train_loss = statistics.mean(train_loss_arr) * 100
avg_val_accuracy = statistics.mean(val_accuracy_arr) * 100
avg_val_loss = statistics.mean(val_loss_arr) * 100

display(f"Average training accuracy: {avg_train_accuracy:.2f}%")
display(f"Average training loss: {avg_train_loss:.2f}%")
display(f"Average validation accuracy: {avg_val_accuracy:.2f}%")
display(f"Average validation loss: {avg_val_loss:.2f}%")

'Average training accuracy: 91.30%'

'Average training loss: 23.64%'

'Average validation accuracy: 90.48%'

'Average validation loss: 25.97%'